# 04 — Transformers

## Contexto

Cuarto y último notebook de modelos. IF y EIF (notebooks 02 y 03) son modelos de árboles: aíslan
cada observación mediante particiones del espacio de features y usan la profundidad de aislamiento
como score de anomalía. Aquí se prueba un enfoque completamente distinto: un **autoencoder basado
en Transformer**, entrenado también solo con datos sanos, que aprende a **reconstruir** el
espectro de una ventana normal. La idea de detección de anomalías es la misma en espíritu que IF/
EIF (aprender "normalidad" sin ver fallos), pero el mecanismo es otro: en vez de aislar por
particiones, el modelo comprime y reconstruye; una ventana cuyo patrón espectral no se parece a lo
visto en entrenamiento se reconstruye peor, y ese error de reconstrucción es el score de anomalía.

## Arquitectura

Cada ventana (vector de features de longitud fija: 603 eléctrica / 5005 vibración / 5608 híbrida)
se trocea en **parches** de tamaño fijo (análogo a un Vision Transformer, pero en 1D). Cada parche
se proyecta a un embedding, se le suma una codificación posicional, y una pila de capas
`TransformerEncoder` (self-attention) procesa la secuencia de parches — permitiendo que el modelo
relacione partes lejanas del espectro entre sí (p. ej. armónicos a distintas frecuencias), algo
que ni IF ni EIF hacen explícitamente. Un decoder lineal reconstruye cada parche. Ver
`utils/torch_model.py` para la implementación completa (`TransformerAutoencoder`).

## Qué se hace

Igual que en 02 y 03: se entrena y evalúa sobre las tres variantes de fuente de señal, solo con
datos sanos. Aquí la búsqueda de hiperparámetros (Optuna, pocos trials por el coste de entrenar
redes neuronales) ajusta la arquitectura (tamaño de parche, dimensión del embedding, nº de capas,
learning rate) minimizando el error de reconstrucción en validación — el criterio estándar de
selección de modelo para autoencoders, distinto del criterio de "compacidad de train" usado en
IF/EIF, pero que persigue el mismo objetivo: un modelo que capture bien qué es "normal".


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import optuna

from utils.config import (
    fijar_semillas, SEED, RUTA_FEATURES, RUTA_RESULTADOS,
    NOMBRES_COLS_ELEC, NOMBRES_COLS_VIB, NOMBRES_COLS, VENTANAS_POR_EXP,
)
from utils.data import cargar_indice
from utils.eval import agregar_por_experimento, agregar_variable, calcular_metricas, guardar_resultado
from utils.torch_model import TransformerAutoencoder, entrenar_autoencoder, error_reconstruccion

fijar_semillas()
torch.manual_seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))

COLS_POR_FUENTE = {
    "electrica": NOMBRES_COLS_ELEC,
    "vibracion": NOMBRES_COLS_VIB,
    "hibrida":   NOMBRES_COLS,
}

FALLOS_BARRA_ROTA = {"e", "b", "v", "p"}
FALLOS_RODAMIENTO = {"g", "o", "r", "c"}

def familia(maquina):
    if maquina == "h":
        return "sano"
    if maquina in FALLOS_BARRA_ROTA:
        return "barra_rota"
    if maquina in FALLOS_RODAMIENTO:
        return "rodamiento"
    return "otro"

index = cargar_indice()

def cargar_split(split, nombre_csv, cols):
    ruta = os.path.join(RUTA_FEATURES, split, nombre_csv)
    return pd.read_csv(ruta, usecols=cols)[cols].values.astype(np.float32)


## Función de entrenamiento + evaluación (reutilizable para las 3 variantes)

- `objective(trial)`: Optuna elige `patch_size`, `d_model`, `num_layers` y `lr`, y minimiza el
  loss de reconstrucción en validación (early stopping incluido en `entrenar_autoencoder`).
  Búsqueda con presupuesto reducido de épocas (recorte de coste); el modelo final se re-entrena
  con más épocas y los mejores hiperparámetros.
- El umbral de anomalía es el percentil 95 del error de reconstrucción de los experimentos sanos
  de train (agregado por mediana) — mismo espíritu que en IF/EIF, aunque aquí no se optimiza junto
  con la arquitectura para no mezclar dos criterios de selección distintos en la misma búsqueda.


In [ ]:
def pipeline_transformer(fuente, n_trials=8, epochs_busqueda=60, epochs_final=250):
    cols = COLS_POR_FUENTE[fuente]
    n_features = len(cols)
    print(f"\n{'='*60}\nFUENTE: {fuente}  ({n_features} features)\n{'='*60}")

    X_train = cargar_split("train", "sano_train.csv", cols)
    X_val   = cargar_split("val",   "sano.csv",        cols)

    def objective(trial):
        patch_size = trial.suggest_categorical("patch_size", [32, 64, 128])
        d_model    = trial.suggest_categorical("d_model", [32, 64, 128])
        num_layers = trial.suggest_int("num_layers", 1, 3)
        lr         = trial.suggest_float("lr", 1e-4, 5e-3, log=True)

        torch.manual_seed(SEED)
        modelo = TransformerAutoencoder(
            n_features=n_features, patch_size=patch_size, d_model=d_model,
            nhead=4, num_layers=num_layers, dim_feedforward=2 * d_model,
        )
        _, val_loss = entrenar_autoencoder(
            modelo, X_train, X_val, epochs=epochs_busqueda, lr=lr,
            paciencia=8, device=DEVICE, verbose=False,
        )
        return val_loss

    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_trial
    print(f"Mejores hiperparametros: {best.params}  (val_loss={best.value:.4f})")

    torch.manual_seed(SEED)
    modelo = TransformerAutoencoder(
        n_features=n_features, patch_size=best.params["patch_size"],
        d_model=best.params["d_model"], nhead=4, num_layers=best.params["num_layers"],
        dim_feedforward=2 * best.params["d_model"],
    )
    modelo, _ = entrenar_autoencoder(
        modelo, X_train, X_val, epochs=epochs_final, lr=best.params["lr"],
        paciencia=20, device=DEVICE, verbose=True,
    )

    scores_train = error_reconstruccion(modelo, X_train, device=DEVICE)
    medianas_train = agregar_por_experimento(scores_train, VENTANAS_POR_EXP)
    umbral = float(np.percentile(medianas_train, 95))
    print(f"Umbral (percentil 95 de sanos train): {umbral:.5f}")

    filas = []
    for m in medianas_train:
        filas.append({"grupo": "sano_train", "maquina": "h", "familia": "sano", "score": m})

    scores_val = error_reconstruccion(modelo, X_val, device=DEVICE)
    for m in agregar_por_experimento(scores_val, VENTANAS_POR_EXP):
        filas.append({"grupo": "sano_val", "maquina": "h", "familia": "sano", "score": m})

    carpeta_test = os.path.join(RUTA_FEATURES, "test")
    for archivo_csv in sorted(os.listdir(carpeta_test)):
        nombre_fallo = archivo_csv.replace(".csv", "")
        grupo_idx = index[(index["Split"] == "test") & (index["Fallo"] == nombre_fallo)]
        n_archivos = len(grupo_idx)
        if n_archivos == 0:
            continue
        maquina = grupo_idx["Maquina"].iloc[0]

        X_grupo = cargar_split("test", archivo_csv, cols)
        scores = error_reconstruccion(modelo, X_grupo, device=DEVICE)
        for m in agregar_variable(scores, n_archivos):
            filas.append({
                "grupo": nombre_fallo, "maquina": maquina,
                "familia": familia(maquina), "score": m,
            })

    df_scores = pd.DataFrame(filas)

    metricas_por_familia = {}
    for fam in ["barra_rota", "rodamiento"]:
        subset = df_scores[df_scores["familia"].isin(["sano", fam])]
        y_true = (subset["familia"] == fam).astype(int).values
        metricas_por_familia[fam] = calcular_metricas(y_true, subset["score"].values, umbral)
        print(f"  {fam:12s} -> precision={metricas_por_familia[fam]['precision']:.2f} "
              f"recall={metricas_por_familia[fam]['recall']:.2f} "
              f"f1={metricas_por_familia[fam]['f1']:.2f} auc={metricas_por_familia[fam]['auc']:.3f}")

    y_true_global = (df_scores["familia"] != "sano").astype(int).values
    metricas_global = calcular_metricas(y_true_global, df_scores["score"].values, umbral)
    print(f"  {'global':12s} -> precision={metricas_global['precision']:.2f} "
          f"recall={metricas_global['recall']:.2f} f1={metricas_global['f1']:.2f} "
          f"auc={metricas_global['auc']:.3f}")

    fig, ax = plt.subplots(figsize=(14, 6))
    orden = ["sano_train", "sano_val"] + sorted(
        g for g in df_scores["grupo"].unique() if g not in ("sano_train", "sano_val")
    )
    df_scores.boxplot(column="score", by="grupo", ax=ax, positions=range(len(orden)))
    ax.axhline(umbral, color="red", linestyle="--", label=f"umbral={umbral:.4f}")
    ax.set_xticklabels(orden, rotation=60, ha="right", fontsize=7)
    ax.set_title(f"Transformer autoencoder — {fuente} — error de reconstrucción por grupo")
    plt.suptitle("")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"resultados/04_boxplot_transformer_{fuente}.png", dpi=120)
    plt.show()

    resultado = {
        "modelo": "transformer_autoencoder",
        "fuente_senal": fuente,
        "n_features": n_features,
        "mejores_hiperparametros": best.params,
        "umbral": umbral,
        "metricas_por_familia": metricas_por_familia,
        "metricas_global": metricas_global,
    }
    path = guardar_resultado(resultado, f"transformer_{fuente}.json")
    print(f"Guardado: {path}")
    return resultado


## Ejecución sobre las 3 variantes de fuente de señal

In [ ]:
resultados_transformer = {}
for fuente in ["electrica", "vibracion", "hibrida"]:
    resultados_transformer[fuente] = pipeline_transformer(fuente)


## Tabla resumen

In [ ]:
resumen = pd.DataFrame([
    {
        "fuente": fuente,
        "auc_barra_rota": r["metricas_por_familia"]["barra_rota"]["auc"],
        "auc_rodamiento": r["metricas_por_familia"]["rodamiento"]["auc"],
        "f1_global": r["metricas_global"]["f1"],
        "auc_global": r["metricas_global"]["auc"],
    }
    for fuente, r in resultados_transformer.items()
])
resumen


## Conclusiones parciales

- El autoencoder basado en Transformer no necesita, igual que IF/EIF, ningún supuesto sobre el
  mecanismo físico del fallo ni sobre el régimen de control de la máquina: aprende directamente de
  los datos sanos.
- A diferencia de IF/EIF, que evalúan cada ventana de forma independiente en cada árbol, la
  self-attention del Transformer puede relacionar partes distintas del espectro entre sí dentro de
  la misma ventana — en principio una ventaja cuando el patrón de fallo involucra relaciones entre
  bandas de frecuencia alejadas (p. ej. varios armónicos simultáneamente), aunque con solo 18
  experimentos sanos de entrenamiento el margen para aprovechar esa capacidad es limitado y el
  riesgo de sobreajuste es mayor que en IF/EIF.
- La tabla anterior, junto con las de los notebooks 02 y 03, alimenta directamente la comparación
  final del notebook 05.
